In [20]:
# from ray.tune.analysis     import ExperimentAnalysis
# from   ray.tune.schedulers import ASHAScheduler
# import ray.cloudpickle     as pickle
# from   ray import tune
# from   ray import train
# from ray.train import Checkpoint, get_checkpoint
from train import GenerateModel, federate_model
from torch.utils.data import DataLoader, TensorDataset
from metrics import eval_model, compare_metric
from functools import partial
from tqdm import tqdm
import torch
import os

import datetime

In [4]:
def load_data(type):
    working_dir  = "/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS" 
    X_val  = torch.load(os.path.join(working_dir,"Data",f"X_{type}.pt"))
    Y_val  = torch.load(os.path.join(working_dir,"Data",f"Y_{type}.pt"))
    return DataLoader(TensorDataset(X_val,Y_val),batch_size=32,shuffle=False)

In [15]:
# working_dir = os.path.split(os.getcwd())[0]
# working_dir = "/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS" 
working_dir = "/Users/drew/Documents/GitHub/FL-with-MIMIC/Replicating Mullenbach/AWS/S3_Bucket"
X_test = torch.load(os.path.join(working_dir,"Data","X_test.pt"))
Y_test = torch.load(os.path.join(working_dir,"Data","Y_test.pt"))
test_loader = DataLoader(TensorDataset(X_test,Y_test),batch_size=32,shuffle=False)

In [13]:
logging_path = os.path.join(working_dir,"S3_Bucket", "logs","logger.log")
low_ge_path  = os.path.join(working_dir,"S3_Bucket", "logs","low_ge.pt")
max_auc_path = os.path.join(working_dir,"S3_Bucket", "logs","max_auc.pt")
model_path   = os.path.join(working_dir,"S3_Bucket")
os.path.isfile(logging_path)

True

In [5]:
def log_detail(desc: str, file: str):
    with open(file, mode = 'a') as f:
        time_now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f'{time_now}: {desc}\n')

In [ ]:
######################################## CONFIG
config = {
        "batch_size" : 16,
        "lr"         : 0.000365, #0.0001,
        "n_filters"  : 16,
        "window_size": 3,
        "epochs"     : 4,
        "rounds"     : 1
    }
######################################## Global Model
# working_dir  = "/home/drew/FL-with-MIMIC/Replicating Mullenbach/AWS" 
working_dir = '/Users/drew/Documents/GitHub/FL-with-MIMIC/Replicating Mullenbach/processed_full.w2v'
model = GenerateModel(table_path = working_dir, # os.path.join(working_dir,"Model","processed_full.w2v"),
                    num_of_filters = config['n_filters'],
                    kernel_size    = config['window_size'])


In [9]:
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device("cpu")
device

device(type='mps')

In [ ]:
########################################
min_error     = float('inf')
max_auc_macro = 0
########################################
for round in tqdm(range(3_000), colour = 'blue', desc = 'Federated Training'):
    trained_model = federate_model(config)
    model.load_state_dict(trained_model)
    ######################################## Generalization Error
    train_hist = eval_model(
               model       = model,
               device      = torch.device("cuda"),
               data_loader = load_data(type = 'train')
    )
    val_hist = eval_model(
               model       = model,
               device      = torch.device("cuda"),
               data_loader = load_data(type = 'val')
    )
    ######################################## Save the best if it exists
    # torch.save(model.state_dict(),)
    ge_error = abs(train_hist['auc_macro'] - val_hist['auc_macro'])
    if min_error > ge_error:
        log_detail(desc = f'Min: {min_error:,.4f}->{ge_error:,.4f};{round}',file = logging_path)
        min_error = ge_error
        torch.save(model.state_dict(), f = low_ge_path)
    
    if val_hist['auc_macro'] > max_auc_macro:
        log_detail(desc = f'Min: {max_auc_macro:,.4f}->{val_hist['auc_macro']:,.4f};{round}',file = logging_path)
        max_auc_macro = val_hist['auc_macro']
        torch.save(model.state_dict(), f = max_auc_path)

Round Training: 100%|██████████| 1/1 [00:08<00:00,  8.98s/it]


____

In [17]:
model_path = "/Users/drew/Documents/GitHub/FL-with-MIMIC/Replicating Mullenbach/AWS/S3_Bucket/logs/max_auc.pt"
state_dict = torch.load(f = model_path,map_location=torch.device("cpu"), weights_only=True)
model.load_state_dict(state_dict)
model.to(torch.device("cpu"))

ConvAttnPool(
  (embed): Embedding(150854, 100, padding_idx=150853)
  (conv): Conv1d(100, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (U): Linear(in_features=16, out_features=50, bias=True)
  (final): Linear(in_features=16, out_features=50, bias=True)
  (embed_drop): Dropout(p=0.2, inplace=False)
)

In [18]:
hist = eval_model(model   = model,
           device  = torch.device('cpu'),
           data_loader = test_loader)

In [ ]:
# tune_results = os.path.join(os.path.split(os.getcwd())[0],"S3_Bucket", "results")
# print(tune_results)
# folders = os.listdir(tune_results)
# print(folders)
# trials = ExperimentAnalysis(os.path.join(tune_results,folders[4]))

In [ ]:
# trials.get_best_trial(metric = 'auc_micro', mode = 'max').config

In [ ]:
# columns = [
#     "auc_macro",
#     "auc_micro",
#     "f1_macro",
#     "f1_micro",
#     "time_since_restore",
#     "config/lr",
#     "config/n_filters",
#     "config/window_size",
#     "config/epochs",
#     "config/batch_size"
# ]

In [ ]:
# import pandas as pd
# df: pd.DataFrame = trials.dataframe()
# df[columns].sort_values(by = 'auc_macro',ascending=False)

In [22]:
benchmark = {
    'auc_macro' : 0.884,
    'auc_micro' : 0.916,
    'f1_macro'  : 0.576,
    'f1_micro'  : 0.633
}

for key in benchmark.keys():
    print(compare_metric(benchmark,hist,key))

auc_macro: -20.48% change (0.8840 → 0.7029)
auc_micro: -18.63% change (0.9160 → 0.7454)
f1_macro: -52.48% change (0.5760 → 0.2737)
f1_micro: -50.18% change (0.6330 → 0.3153)
